## Silver paths from ADLS

In [0]:
matches = "abfss://<container>@<storage-account>.dfs.core.windows.net/<env>/<user>/<project>/silver/matches/singles/"
players = "abfss://<container>@<storage-account>.dfs.core.windows.net/<env>/<user>/<project>/silver/players/"
rankings= "abfss://<container>@<storage-account>.dfs.core.windows.net/<env>/<user>/<project>/silver/rankings/"

## Samples of matches(singles), players, rankings

In [0]:
matches = spark.read.format("delta").load(matches).limit(5).display()
players = spark.read.format("delta").load(players).limit(5).display()
rankings = spark.read.format("delta").load(rankings).limit(5).display()

tourney_id,tourney_name,surface,draw_size,tourney_level,tourney_date,match_year,round,winner_id,loser_id,winner_rank,loser_rank,score,best_of,match_type
1991-339,Adelaide,Hard,32,A,1990-12-31,1990,R32,101723,101414,56,2,6-4 3-6 7-6(2),3,singles
1991-339,Adelaide,Hard,32,A,1990-12-31,1990,R32,100946,101256,304,75,6-3 3-6 7-6(6),3,singles
1991-339,Adelaide,Hard,32,A,1990-12-31,1990,R32,101234,101421,82,69,6-0 6-4,3,singles
1991-339,Adelaide,Hard,32,A,1990-12-31,1990,R32,101889,101703,50,84,7-6(2) 6-1,3,singles
1991-339,Adelaide,Hard,32,A,1990-12-31,1990,R32,101274,101843,88,28,7-5 6-3,3,singles


player_id,name_first,name_last,hand,dob,ioc,height,wikidata_id
100001,Gardnar,Mulloy,R,null,USA,185,Q54544
100002,Pancho,Segura,R,null,ECU,168,Q54581
100003,Frank,Sedgman,R,null,AUS,180,Q962049
100004,Giuseppe,Merlo,R,null,ITA,null,Q1258752
100005,Richard,Gonzalez,R,null,USA,188,Q53554


ranking_date,rank,player,points,ranking_year
1973-08-27,129,100005,null,1973
1973-08-27,114,100011,null,1973
1973-08-27,6,100016,null,1973
1973-08-27,19,100022,null,1973
1973-08-27,82,100025,null,1973


## Gold: fact_singles_matches

Business Concept
- **Grain:** One row per singles match  
- **Purpose:** Analytics on match outcomes, player performance, tournament details, and surfaces

Enhancements
- Join players for clean names and attributes (full name, hand, height, IOC code)  
- Join rankings for winner and loser at the match date  
- Add derived metrics to facilitate analytics  
- Normalize winner and loser into meaningful metrics

Example Derived Columns
- **winner_is_left_handed:** Flag indicating if the winner is left-handed  
- **loser_is_left_handed:** Flag indicating if the loser is left-handed  
- **rank_diff:** Difference between loser rank and winner rank  
- **is_upset:** Flag indicating if a lower-ranked player beat a higher-ranked player  

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Load Silver Tables
matches_df = spark.read.format("delta").load(matches)
players_df = spark.read.format("delta").load(players)
rankings_df = spark.read.format("delta").load(rankings)

# Prepare Player Dimension
players_dim = players_df.select(
    F.col("player_id"),
    F.concat_ws(" ", "name_first", "name_last").alias("full_name"),
    "hand", "ioc", "height" 
)

# Enrich Matches with Player Info
matches_enriched = (
    matches_df
    # Winner info
    .join(players_dim.withColumnRenamed("player_id", "winner_id") \
                    .withColumnRenamed("full_name", "winner_full_name") \
                    .withColumnRenamed("hand", "winner_hand") \
                    .withColumnRenamed("height", "winner_height") \
                    .withColumnRenamed("ioc", "winner_ioc"),
          on="winner_id", how="left")
    # Loser info
    .join(players_dim.withColumnRenamed("player_id", "loser_id") \
                    .withColumnRenamed("full_name", "loser_full_name") \
                    .withColumnRenamed("hand", "loser_hand") \
                    .withColumnRenamed("height", "loser_height") \
                    .withColumnRenamed("ioc", "loser_ioc"),
          on="loser_id", how="left")
)

# Prepare Rankings Snapshots
# Use window to get latest ranking
ranking_window = Window.partitionBy("player").orderBy(F.col("ranking_date").desc())

rankings_latest = rankings_df.withColumn("rn", F.row_number().over(ranking_window)) \
                             .filter("rn = 1").drop("rn")

# Rename columns BEFORE joining to avoid conflicts
winner_rankings = rankings_latest.withColumnRenamed("player", "winner_id") \
                                .withColumnRenamed("rank", "winner_rank") \
                                .withColumnRenamed("points", "winner_rank_points") \
                                .withColumnRenamed("ranking_date", "winner_ranking_date") \
                                .withColumnRenamed("ranking_year", "winner_ranking_year")

loser_rankings = rankings_latest.withColumnRenamed("player", "loser_id") \
                               .withColumnRenamed("rank", "loser_rank") \
                               .withColumnRenamed("points", "loser_rank_points") \
                               .withColumnRenamed("ranking_date", "loser_ranking_date") \
                               .withColumnRenamed("ranking_year", "loser_ranking_year")

# Drop existing ranking columns from matches_enriched to prevent ambiguity
matches_enriched = matches_enriched.drop("winner_rank", "loser_rank",
                                         "winner_rank_points", "loser_rank_points")

# Join Rankings to Matches
matches_gold = (
    matches_enriched
    .join(winner_rankings, on="winner_id", how="left")
    .join(loser_rankings, on="loser_id", how="left")
)

# Derived Metrics
matches_gold = matches_gold.withColumn(
    "rank_diff", F.col("loser_rank") - F.col("winner_rank")
).withColumn(
    "is_upset", F.col("winner_rank") > F.col("loser_rank")
).withColumn(
    "winner_is_left_handed", F.when(F.col("winner_hand") == "L", 1).otherwise(0)
).withColumn(
    "loser_is_left_handed", F.when(F.col("loser_hand") == "L", 1).otherwise(0)
)

matches_gold.limit(5).display()

# Save Gold Table to ADLS as Delta
gold_path = "abfss://<container>@<storage-account>.dfs.core.windows.net/<env>/<user>/<project>/gold/fact_singles_matches/"

matches_gold.write.format("delta") \
    .mode("overwrite") \
    .save(gold_path)

loser_id,winner_id,tourney_id,tourney_name,surface,draw_size,tourney_level,tourney_date,match_year,round,score,best_of,match_type,winner_full_name,winner_hand,winner_ioc,winner_height,loser_full_name,loser_hand,loser_ioc,loser_height,winner_ranking_date,winner_rank,winner_rank_points,winner_ranking_year,loser_ranking_date,loser_rank,loser_rank_points,loser_ranking_year,rank_diff,is_upset,winner_is_left_handed,loser_is_left_handed
101414,101723,1991-339,Adelaide,Hard,32,A,1990-12-31,1990,R32,6-4 3-6 7-6(2),3,singles,Magnus Larsson,R,SWE,193,Boris Becker,R,GER,190,2004-03-29,1422,1,2004,2000-07-03,257,125,2000,-1165,true,0,0
101256,100946,1991-339,Adelaide,Hard,32,A,1990-12-31,1990,R32,6-3 3-6 7-6(6),3,singles,Slobodan Zivojinovic,R,YUG,198,Mark Kratzmann,L,AUS,178,1992-11-02,455,34,1992,1993-07-12,1147,1,1993,692,false,0,1
101421,101234,1991-339,Adelaide,Hard,32,A,1990-12-31,1990,R32,6-0 6-4,3,singles,Patrik Kuhnen,R,GER,190,Veli Paloheimo,R,FIN,183,1998-07-13,1341,1,1998,1994-07-18,1194,1,1994,-147,true,0,0
101703,101889,1991-339,Adelaide,Hard,32,A,1990-12-31,1990,R32,7-6(2) 6-1,3,singles,Todd Woodbridge,R,AUS,178,Guillaume Raoux,R,FRA,180,2002-09-30,1151,3,2002,2001-07-02,1033,4,2001,-118,true,0,0
101843,101274,1991-339,Adelaide,Hard,32,A,1990-12-31,1990,R32,7-5 6-3,3,singles,Udo Riglewski,R,GER,185,Sergi Bruguera,R,ESP,188,1996-01-22,1274,1,1996,2003-07-28,668,19,2003,-606,true,0,0


# Gold: dim_players

Business Concept
- **Grain:** One row per player
- **Purpose:** Store player attributes for analytics, reporting, and joining with match and ranking data

Enhancements
- Concatenate first and last name into a clean `full_name`  
- Add `is_left_handed` flag for analytics on player handedness  
- Include physical and metadata attributes: height, hand, IOC code, date of birth, wikidata_id  

Example Derived Columns
- **full_name:** Concatenation of first and last name  
- **is_left_handed:** Flag indicating if the player is left-handed


In [0]:
# Load Silver Players table
players_df = spark.read.format("delta").load(players)

# Create dim_players for Gold Layer
dim_players = players_df.select(
    F.col("player_id"),
    F.concat_ws(" ", "name_first", "name_last").alias("full_name"),
    "hand",
    "ioc",
    "height",
    "dob",
    "wikidata_id"
).withColumn(
    "is_left_handed", F.when(F.col("hand") == "L", 1).otherwise(0)
)

# Display dim_players
dim_players.limit(5).display()

# Save dim_players as Delta (Gold Layer)
dim_players_path = "abfss://<container>@<storage-account>.dfs.core.windows.net/<env>/<user>/<project>/gold/dim_players/"

dim_players.write.format("delta") \
    .mode("overwrite") \
    .save(dim_players_path)

player_id,full_name,hand,ioc,height,dob,wikidata_id,is_left_handed
100001,Gardnar Mulloy,R,USA,185,null,Q54544,0
100002,Pancho Segura,R,ECU,168,null,Q54581,0
100003,Frank Sedgman,R,AUS,180,null,Q962049,0
100004,Giuseppe Merlo,R,ITA,null,null,Q1258752,0
100005,Richard Gonzalez,R,USA,188,null,Q53554,0


#Gold: fact_player_rankings

Business Concept
- **Grain:** One row per player per ranking_date
- **Purpose:** Track player ranking and points trends over time for performance analytics

Enhancements
- Compute `rank_change` compared to the previous ranking date  
- Compute `points_change` compared to the previous ranking date  
- Enable trend analysis and rolling averages for ranking and points  
- Support joining with matches and players for richer analytics

Example Derived Columns
- **rank_change:** Difference between previous rank and current rank  
- **points_change:** Difference between previous points and current points

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Load Silver Rankings table
rankings_df = spark.read.format("delta").load(rankings)

# Define window to compute previous rank/points for delta metrics
player_window = Window.partitionBy("player").orderBy("ranking_date")

# Create fact_player_rankings for Gold Layer
fact_player_rankings = rankings_df.withColumn(
    "prev_rank", F.lag("rank").over(player_window)
).withColumn(
    "rank_change", F.col("prev_rank") - F.col("rank")
).withColumn(
    "prev_points", F.lag("points").over(player_window)
).withColumn(
    "points_change", F.col("points") - F.col("prev_points")
)

# Display fact_player_rankings
fact_player_rankings.limit(5).display()

# Save fact_player_rankings as Delta (Gold Layer)
fact_player_rankings_path = "abfss://<container>@<storage-account>.dfs.core.windows.net/<env>/<user>/<project>/gold/fact_player_rankings/"

fact_player_rankings.write.format("delta") \
    .mode("overwrite") \
    .save(fact_player_rankings_path)

ranking_date,rank,player,points,ranking_year,prev_rank,rank_change,prev_points,points_change
1974-06-03,274,100003,null,1974,null,null,null,null
1974-08-12,306,100003,null,1974,274,-32,null,null
1974-09-09,312,100003,null,1974,306,-6,null,null
1974-09-30,282,100003,null,1974,312,30,null,null
1974-11-11,297,100003,null,1974,282,-15,null,null
